In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

Mounted at /content/drive
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [2]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"

!pip install arch

Agent pid 847
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-aae3c6b
# github.com:22 SSH-2.0-aae3c6b
# github.com:22 SSH-2.0-aae3c6b
# github.com:22 SSH-2.0-aae3c6b
# github.com:22 SSH-2.0-aae3c6b
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 52.2 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

from functools import cached_property
from typing import Union, Optional, Dict

from scipy.stats import norm, t as tdist
from scipy.stats import chi2


try:
    from arch import arch_model
    HAS_ARCH = True
except Exception:
    HAS_ARCH = False


#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)


# （建議）再開啟 XLA 與混合精度
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("是否可用GPU:", tf.test.is_gpu_available())
print("使用中的裝置:", tf.config.list_physical_devices('GPU'))

print("✓ GPU 初始化流程完成")


# 1. 定義檔案路徑
file_paths = {
    # "bonds_day": "./filtered_output/bonds_day_clean_period.csv",
    # "bonds_hour": "./filtered_output/bonds_hour_clean_period.csv",
    # "crypto_day": "./filtered_output/crypto_day_clean_period.csv",
    # "crypto_hour": "./filtered_output/crypto_hour_clean_period.csv",
    # "others_day": "./filtered_output/others_day_clean_period.csv",
    # "others_hour": "./filtered_output/others_hour_clean_period.csv",
    "stock_day":  "./filtered_output/stock_day_fluctuation_aligned.csv"
    # "stock_hour": "./filtered_output/stock_hour_clean_period.csv"
}


def find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df



raw_dfs = {}
for name, path in file_paths.items():
    raw_dfs[name] = read_and_clean(path)
    # print(raw_dfs[name].columns)

# ===== 資料結構：取代 all_data =====
class AssetGroupLite:
    def __init__(self, name, df):
        self.name = name
        df = df.copy()
        df['DATE'] = pd.to_datetime(df['DATE'])
        self.raw = df.set_index('DATE').sort_index()

    @cached_property
    def _close_cols(self):
        return [c for c in self.raw.columns if c.endswith('_CLOSE')]

    @cached_property
    def _vol_cols(self):
        return [c for c in self.raw.columns if c.endswith('_VOLUME')]

    @cached_property
    def close_ln(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return np.log(self.raw[self._close_cols]).rename(
            columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln')
        )

    @cached_property
    def close_ln_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        logp = np.log(self.raw[self._close_cols])
        return (
            logp.diff()
               .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln_ret'))
               .dropna(how='all')
        )

    @cached_property
    def close_arith_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return (
            self.raw[self._close_cols].pct_change()
                .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_arith_ret'))
                .dropna(how='all')
        )

class DataRepository:
    REQUIRED_INDEX = "DATE"

    def __init__(self, raw_dfs: dict, check_schema: bool = True):
        self.groups = {}
        for name, df in raw_dfs.items():
            if check_schema:
                assert 'DATE' in df.columns, f"{name}: 缺少 DATE 欄"
            self.groups[name] = AssetGroupLite(name, df)

    # 補上 group()，方便外部與內部呼叫
    def group(self, name: str) -> AssetGroupLite:
        if name not in self.groups:
            raise KeyError(f"Group '{name}' 不存在。可用群組：{list(self.groups.keys())}")
        return self.groups[name]

    def series(self, group: str, series_name: str) -> pd.Series:
        g = self.group(group)
        # 加上 raw → 能抓 OHLCV、IS_TRADING 等原始欄
        search_order = ['close_ln_ret', 'close_arith_ret', 'close_ln', 'raw']

        # 1) 直接命中
        for key in search_order:
            tbl = getattr(g, key)
            if series_name in tbl.columns:
                return tbl[series_name]

        # 2) 容錯：只給 base symbol，自動補候選
        base = (series_name
                .replace('_CLOSE', '')
                .replace('_OPEN', '')
                .replace('_HIGH', '')
                .replace('_LOW', '')
                .replace('_VOLUME', '')
                .replace('_IS_TRADING', '')
                .replace('_CLOSE_ln_ret', '')
                .replace('_CLOSE_arith_ret', '')
                .replace('_CLOSE_ln', ''))
        candidates = [
            f'{base}_CLOSE_ln_ret',
            f'{base}_CLOSE_arith_ret',
            f'{base}_CLOSE_ln',
            f'{base}_OPEN',
            f'{base}_HIGH',
            f'{base}_LOW',
            f'{base}_CLOSE',
            f'{base}_VOLUME',
            f'{base}_IS_TRADING'
        ]
        for cand in candidates:
            for key in search_order:
                tbl = getattr(g, key)
                if cand in tbl.columns:
                    return tbl[cand]

        raise KeyError(f"{group}: 找不到 {series_name} 或候選 {candidates}")

    # 取整張表
    def table(self, group: str, table_name: str) -> pd.DataFrame:
        g = self.group(group)
        if not hasattr(g, table_name):
            raise KeyError(f"{group}: 無表 '{table_name}'。可用表：['close_ln_ret','close_arith_ret','close_ln']")
        return getattr(g, table_name)

    # 若要直接拿 raw 的原始價/量欄位（例如 *_CLOSE 或 *_VOLUME）
    def raw_series(self, group: str, raw_col: str) -> pd.Series:
        g = self.group(group)
        if raw_col not in g.raw.columns:
            raise KeyError(f"{group}: raw 中沒有欄位 {raw_col}")
        return g.raw[raw_col]

repo = DataRepository(raw_dfs)
print("\n success repo")

Num GPUs: 1 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ memory growth set
是否可用GPU: True
使用中的裝置: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU 初始化流程完成

 success repo


In [5]:
# ------------------ Features ------------------
def build_feature_df(repo: DataRepository, group: str, symbol: str) -> pd.DataFrame:
    s_open  = repo.raw_series(group, f'{symbol}_OPEN').asfreq('D')
    s_high  = repo.raw_series(group, f'{symbol}_HIGH').asfreq('D')
    s_low   = repo.raw_series(group, f'{symbol}_LOW').asfreq('D')
    s_close = repo.raw_series(group, f'{symbol}_CLOSE').asfreq('D')
    s_vol   = repo.raw_series(group, f'{symbol}_VOLUME').asfreq('D')
    s_flag = repo.raw_series(group, f'{symbol}_IS_TRADING').asfreq('D')
    s_lnrt  = repo.series(group, f'{symbol}_CLOSE_ln_ret').asfreq('D')
    s_ln  = repo.series(group, f'{symbol}_CLOSE_ln').asfreq('D')
    df = pd.concat([
        s_lnrt.rename(f'{symbol}_LN_RET'),
        s_open.rename(f'{symbol}_OPEN'),
        s_high.rename(f'{symbol}_HIGH'),
        s_low.rename(f'{symbol}_LOW'),
        s_close.rename(f'{symbol}_CLOSE'),
        s_vol.rename(f'{symbol}_VOLUME'),
        s_flag.rename(f'{symbol}_IS_TRADING'),
        s_ln.rename(f'{symbol}_CLOSE_LN')
        ], axis=1)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.dropna(how='any')

############### mcHARCH
def fit_vol_per_window(ret_window, mode='garch'):
    y = ret_window.dropna().astype(float)
    if len(y) < 30:
        lam = 0.94
        ewma_var = y.pow(2).ewm(alpha=1-lam, adjust=False).mean()
        log_sigma_series = 0.5 * np.log(np.maximum(ewma_var.values, 1e-12))
        log_sigma_series = pd.Series(log_sigma_series, index=ewma_var.index)\
                              .reindex(ret_window.index).ffill().bfill()
        sigma_next = np.sqrt(lam * ewma_var.iloc[-1] + (1-lam) * y.iloc[-1]**2)
        return log_sigma_series, float(sigma_next)

    vol = 'HARCH' if mode.lower() == 'harch' else 'GARCH'
    p, q = (3, 0) if vol == 'HARCH' else (1, 1)

    scale = 100.0
    am = arch_model(y.values * scale, mean='Zero', vol=vol, p=p, q=q, dist='t')
    res = am.fit(disp='off')

    # 視窗內「過濾」波動：先除回 scale，再做下限保護
    sigma_series = res.conditional_volatility          # ndarray
    sigma_series = np.maximum(sigma_series / scale, 1e-12)
    log_sigma_series = np.log(sigma_series)
    log_sigma_series = pd.Series(log_sigma_series, index=y.index)\
                          .reindex(ret_window.index).ffill().bfill()

    # 一步前瞻：variance → sqrt → 除回 scale
    fvar = res.forecast(horizon=1, reindex=False).variance.values[-1, 0]
    sigma_next = float(np.sqrt(fvar) / scale)

    return log_sigma_series, sigma_next

################## 風險
def compute_var_es(mu, sigma, alpha,
                   dist=None,              # 'normal' 或 't'；若為 None 則依 nu 推斷
                   nu=None,                # dist='t' 時需要；dist='normal' 時忽略
                   price_base=None,        # 若給，會同時輸出價格層 VaR/ES
                   standardized_t=True     # t 分布是否以 Var=1 標準化
                   ):
    # """
    # 計算左尾 VaR/ES（報酬層 + 價格層）。
    # - mu, sigma: 可為 Series 或純量；若 Series，index 會自動對齊。
    # - alpha: 左尾機率（例如 0.05、0.01）。
    # - dist: 'normal' 或 't'。若 None：若 nu 是 None → 'normal'；否則 → 't'。
    # - nu:    t 分布自由度（>2）；dist='normal' 時忽略。
    # - standardized_t: True 時採用 Var=1 的 t-innovations（尺度 s = sqrt((nu-2)/nu)）。
    # - price_base: 若給（通常為 P_{t-1}），同時回傳 VaR_price / ES_price。
    # 回傳：DataFrame，含 VaR_ret、ES_ret（與可選 VaR_price、ES_price）。
    # """
    # ----- 推斷分布類型（維持舊版相容） -----
    if dist is None:
        dist = 't' if nu is not None else 'normal'
    dist = dist.lower()
    if dist not in ('normal', 'gaussian', 't', 'student', 'student-t'):
        raise ValueError("dist 必須為 'normal' 或 't'。")

    # ----- 對齊輸入 -----
    mu_s  = pd.Series(mu)
    sig_s = pd.Series(sigma)
    idx   = mu_s.index.union(sig_s.index)
    mu_s  = mu_s.reindex(idx)
    sig_s = sig_s.reindex(idx).clip(lower=0.0)  # 數值保險：σ >= 0
    price_s = pd.Series(price_base).reindex(idx) if price_base is not None else None

    if not (0.0 < alpha < 0.5):
        raise ValueError("alpha 應位於 (0, 0.5) 的左尾區間。")

    # ----- 常態分布 -----
    if dist in ('normal', 'gaussian'):
        z_alpha = norm.ppf(alpha)          # 負值
        phi     = norm.pdf(z_alpha)
        ES_std  = -phi / alpha             # 標準常態左尾 ES（均值0、σ=1）
        VaR_ret = mu_s + sig_s * z_alpha
        ES_ret  = mu_s + sig_s * ES_std

    # ----- t 分布 -----
    else:
        if nu is None:
            raise ValueError("dist='t' 需要提供 nu。")
        nu = float(nu)
        if nu <= 2:
            raise ValueError("t 分布計算 ES 需要 nu > 2（變異數有限）。")

        t_alpha = tdist.ppf(alpha, df=nu)     # 負值
        # 標準化尺度：Var=1 的 t-innovations 需乘 s = sqrt((nu-2)/nu)
        s = np.sqrt((nu - 2.0) / nu) if standardized_t else 1.0
        f_t = tdist.pdf(t_alpha, df=nu)

        # 標準型 t 的左尾 ES（均值0、尺度1）封閉解
        ES_std = - ((nu + t_alpha**2) / ((nu - 1.0) * alpha)) * f_t

        VaR_std = s * t_alpha
        ES_std  = s * ES_std

        VaR_ret = mu_s + sig_s * VaR_std
        ES_ret  = mu_s + sig_s * ES_std

    out = pd.DataFrame({'VaR_ret': VaR_ret, 'ES_ret': ES_ret}, index=idx)

    # ----- 價格層（若提供基準價） -----
    if price_s is not None:
        out['VaR_price'] = price_s * np.exp(out['VaR_ret'])
        out['ES_price']  = price_s * np.exp(out['ES_ret'])

    return out.reindex(mu_s.index)


# === 0) (可選) 判斷 t 殘差是否已標準化 Var≈1 ===
def check_t_standardization(eps, sigma) -> float:
    # """
    # 回傳標準化殘差 z = eps/sigma 的樣本變異。
    # ≈1 代表 standardized_t=True；若 ≈ ν/(ν-2) 代表未標準化。
    # """
    z = np.asarray(eps) / np.maximum(np.asarray(sigma), 1e-12)
    z = z[np.isfinite(z)]
    return float(np.var(z, ddof=1))


# ------------------ Config ------------------
ASSET_SYMBOL_ES1 = 'ES1'
ASSET_SYMBOL_VIX = 'VIX'
GROUP_DAY   = 'stock_day'
TARGET_START_STR = '2005-06-01'
TARGET_END_STR   = '2006-06-30'

GARCH_WINDOW_DAY = 252


feat_ES1 = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_ES1)
feat_VIX = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_VIX)

df_day = feat_ES1.join(feat_VIX, how='left')

# --- 讓索引成為 datetime（很重要） ---
df_day.index = pd.to_datetime(df_day.index, errors='coerce')
df_day = df_day.sort_index()
df_day = df_day.loc[df_day['ES1_IS_TRADING'] == 1]

# print(df_day.head())
# print(df_day.columns)

df_day.to_csv("./filtered_output/df_day.csv", index=True)


PRED_START = pd.to_datetime(TARGET_START_STR)
PRED_END   = pd.to_datetime(TARGET_END_STR)

# 把回測期限制在資料範圍內
data_start = df_day.index.min()
data_end   = df_day.index.max()
if PRED_START < data_start: PRED_START = data_start
if PRED_END   > data_end:   PRED_END   = data_end

# 若 PRED_START 不是可用交易日，推到 >= PRED_START 的第一個交易日
try:
    PRED_START = df_day.index[df_day.index.searchsorted(PRED_START)]
except Exception:
    # 若整段都沒資料，直接報錯
    raise RuntimeError("資料期間與回測期間沒有交集，請調整 TARGET_START/END。")


first_needed = PRED_START - pd.Timedelta(days=GARCH_WINDOW_DAY)
es1_min = df_day.index.min()
ES1_close_series = df_day['ES1_CLOSE']
if es1_min > first_needed:
    raise RuntimeError(f"Insufficient history: need <= {first_needed}, have from {ES1_close_series.index.min()}")



# ------------------ Model ------------------

# 1) 準備母表（確保排序與期間）
# 全歷史的交易日索引（只留 ES1 開市）
dates_all = df_day.index
# print(dates_all)

# 回測區間（仍然要取，決定哪些 t 需要預測）
mask_period = (df_day.index >= PRED_START) & (df_day.index <= PRED_END)
dates_period = df_day.index[mask_period]


# 2) 方便取用的短名
feat_df = df_day  # 與舊程式一致
rows = []


# 3) 走索引的滑窗，不假設日期連續
#   第 i 筆要預測的是 dates[i]（以 t 表示），窗口是 dates[i-WINDOW_DAY : i)（只到 t-1）

for t in dates_period:
    # 1) 找出 t 在「全歷史」中的**位置**（不是在 dates_period 中的 i）
    pos = dates_all.searchsorted(t)   # 或：pos = df_day.index.get_loc(t)
    if pos < GARCH_WINDOW_DAY:
      continue


    garch_idx = np.arange(pos - GARCH_WINDOW_DAY, pos)  # → [t-GARCH,   ..., t-1]

    garch_dates = dates_all.take(garch_idx)

    # 目標日與 t-1
    t_minus_1 = dates_all[pos - 1]   # 直接往前一格 → t-1
    print(f"pos = {dates_all[pos]},  window = {garch_dates[0]} -> {garch_dates[-1]}")

#     print(f"start {t} ")
    # print(f"start {t} | pos {pos} |　t_minus_1 {t_minus_1}")

    # print(f"garch head={list(garch_dates[:5].date)}")
    # print(f"garch tail={list(garch_dates[-5:].date)}\n")
    # print(f"Xw_period head={list(win_dates[:5].date)}")
    # print(f"Xw_period tail={list(win_dates[-5:].date)}\n")


    # --- GARCH（視窗內，只用到 t-1 的 return） ---
    # 你的 feature_names 中若名稱是 ES1_LN_RET 就用它；否則改用你實際欄名
    # garch_dates有300天做完garch後便做zcore存到log_sigma300，再將其取所需時間段出來
    ret_col = 'ES1_LN_RET' if 'ES1_LN_RET' in feat_df.columns else 'LN_RET'
    log_sigma_series, garch_sigma_hat_t = fit_vol_per_window(
        ret_window=feat_df.loc[garch_dates, ret_col], # 只含 ≤t-1
        mode='garch'  # 'harch' 也可
    )
    # print(len(log_sigma_series))

    # --- 真實值 ---
    ln_t   = float(feat_df.loc[t,'ES1_CLOSE_LN'])
    p_t   = float(feat_df.loc[t, 'ES1_CLOSE'])
    p_tm1   = float(feat_df.loc[t_minus_1, 'ES1_CLOSE'])
    r_t   = float(feat_df.loc[t, 'ES1_LN_RET'])


    rows.append({
        'DATE'      : t,
        'price_true': p_t,
        'price_base_aligned': p_tm1,
        'ln_true'   : ln_t,
        'ret_true'  : r_t,
        'sigma_next': garch_sigma_hat_t,
        'mu'        : 0.0
    })


# # # ------------------ Evaluation ------------------
# ######################################################################################################
# # ##########  風險
# # 先建 df_out
df_out = pd.DataFrame(rows).set_index("DATE").sort_index()

# # 1) 直接看前後幾筆（最常用）
# print(df_out.head(10))
# print(df_out.tail(10))

# # 2) 看欄位型態/缺值概況
# print(df_out.info())
# print(df_out.isna().sum())

# # 3) 快速描述統計（檢查 sigma 量級是否正常）
# print(df_out[["ret_true", "sigma_next"]].describe())

# 參數
ALPHAS = [0.05, 0.01]
NU     = 6.0
STD_T  = True

mu_series    = df_out['mu']      # μ̂_t（as-of t-1 對 t）
sigma_series = df_out['sigma_next']    # σ̂_t（as-of t-1 對 t）
price_base   = df_out['price_base_aligned']  # P_{t-1}（對齊到 t）

for a in ALPHAS:
    res = compute_var_es(mu_series, sigma_series, alpha=a,
                     dist='t', nu=NU, price_base=price_base, standardized_t=STD_T)
    tag = f'{int((1 - a)*100):02d}'
    df_out[f'VaR_ret_{tag}'] = res['VaR_ret']
    df_out[f'ES_ret_{tag}']  = res['ES_ret']
    if 'VaR_price' in res.columns:
        df_out[f'VaR_price_{tag}'] = res['VaR_price']
    if 'ES_price' in res.columns:
        df_out[f'ES_price_{tag}']  = res['ES_price']

# 報酬層違規
df_out['viol_95'] = (df_out['ret_true'] < df_out['VaR_ret_95']).astype(int)
df_out['viol_99'] = (df_out['ret_true'] < df_out['VaR_ret_99']).astype(int)

# 快速違規率
viol_rate_95 = float(df_out['viol_95'].mean())
viol_rate_99 = float(df_out['viol_99'].mean())

# 逐日輸出（修正漏引號）
cols_daily = [
    'price_true','price_base_aligned',
    'ret_true','ln_true',
    'sigma_next',   # ← 修正這裡
    'VaR_ret_95','ES_ret_95','viol_95',
    'VaR_ret_99','ES_ret_99','viol_99',
    'VaR_price_95','ES_price_95','VaR_price_99','ES_price_99'
]
df_export = df_out[[c for c in cols_daily if c in df_out.columns]].copy()
print(df_export.head(10))
print(df_export.tail(10))

# 6b) 摘要矩陣（風控報告用）
var_es_matrics = pd.DataFrame({
    'Metric': [
        'Mean_ret_true', 'Std_ret_true',
        'Mean_VaR_ret_95', 'Mean_ES_ret_95',
        'Mean_VaR_ret_99', 'Mean_ES_ret_99',
        'Viol_rate_95', 'Viol_count_95',
        'Viol_rate_99', 'Viol_count_99',
    ],
    'Value': [
        float(df_out['ret_true'].mean()),
        float(df_out['ret_true'].std(ddof=1)),
        float(df_out['VaR_ret_95'].mean()),
        float(df_out['ES_ret_95'].mean()),
        float(df_out['VaR_ret_99'].mean()),
        float(df_out['ES_ret_99'].mean()),
        viol_rate_95, int(df_out['viol_95'].sum()),
        viol_rate_99, int(df_out['viol_99'].sum()),
    ]
})

# === 匯出檔案 ===
output_dir = "./LSTM_diagnostics"
os.makedirs(output_dir, exist_ok=True)
summary_path = os.path.join(output_dir, "var_es_matrics.csv")
var_es_matrics.to_csv(summary_path, index=False)
daily_path = os.path.join(output_dir, "var_es_daily.csv")
df_export.to_csv(daily_path, index=True)  # index=DATE


stem = f"{ASSET_SYMBOL_ES1.lower()}_garch"
fig1 = f"./LSTM_diagnostics/{stem}_VaR_ES_returns_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"

plt.figure(figsize=(12, 6))

# === 圖 1: 報酬 vs VaR/ES (95%) ===
# True return：淺藍色
plt.plot(
    df_out.index, df_out['ret_true'],
    label='True Return',
    color='blue', alpha=0.7, linewidth=0.8
)

# VaR：綠色曲線（你要的「預測的 VaR 曲線」）
plt.plot(
    df_out.index, df_out['VaR_ret_95'],
    label='VaR 95%',
    color='green', linewidth=0.7, alpha=0.8
)

# # ES：保留（若你也想一起改色可再說）
# plt.plot(
#     df_out.index, df_out['ES_ret_95'],
#     label='ES 95%',
#     color='black', linestyle='--', linewidth=0.8, alpha=0.7
# )

# 違規：紅色「點」
viol = df_out['ret_true'] < df_out['VaR_ret_95']
plt.scatter(
    df_out.index[viol], df_out.loc[viol, 'ret_true'],
    color='red', marker='o', s=14, edgecolors='none', alpha=0.9,
    label='Violation'
)

plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel("Time")
plt.ylabel("Log Return")
plt.title(f"{ASSET_SYMBOL_ES1} Return vs VaR/ES (95%)")
plt.legend()
plt.grid(True)

plt.savefig(fig1, dpi=300, bbox_inches='tight')
plt.close()



# # ------------------ Figures (3+1) ------------------
# fig5 = f"./LSTM_diagnostics/{stem}_VaR_ES_price_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
# fig7 = f"./LSTM_diagnostics/{stem}_VaR_ES_scatter_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"



# ############## 風險
# # === 圖 1: 價格 vs VaR/ES (95%) ===
# plt.figure(figsize=(12,6))
# plt.plot(df_out.index, df_out['price_true'],
#          label='True Price', color='black', linewidth=0.8)
# plt.plot(df_out.index, df_out['VaR_price_95'],
#          label='VaR Price 95%', color='red', linewidth=0.5, alpha=0.7)
# plt.plot(df_out.index, df_out['ES_price_95'],
#          label='ES Price 95%', color='darkred', linestyle='--', linewidth=0.8, alpha=0.7)
# plt.fill_between(df_out.index, 0, df_out['VaR_price_95'],
#                  color='red', alpha=0.05)

# plt.xlabel("Time"); plt.ylabel("Price")
# plt.title(f"{ASSET_SYMBOL_ES1} Price vs VaR/ES (95%)")
# plt.legend(); plt.grid(True)
# plt.savefig(fig5, dpi=300); plt.close()


# # === 圖 3: 散點回測圖 (VaR vs True Return, 95%) ===
# plt.figure(figsize=(6,6))
# plt.scatter(df_out['VaR_ret_95'], df_out['ret_true'],
#             alpha=0.4, color='blue', s=10)  # 點數縮小、透明度提高
# plt.plot([df_out['VaR_ret_95'].min(), df_out['VaR_ret_95'].max()],
#          [df_out['VaR_ret_95'].min(), df_out['VaR_ret_95'].max()],
#          color='red', linestyle='--', linewidth=1.0)
# plt.xlabel("VaR_ret_95"); plt.ylabel("True Return")
# plt.title(f"{ASSET_SYMBOL_ES1} VaR Backtesting (95%)")
# plt.grid(True)
# plt.savefig(fig7, dpi=300); plt.close()

# print('Figures:', fig1, fig2, fig3, fig4)

pos = 2005-06-01 00:00:00,  window = 2004-06-07 00:00:00 -> 2005-05-31 00:00:00
pos = 2005-06-02 00:00:00,  window = 2004-06-08 00:00:00 -> 2005-06-01 00:00:00
pos = 2005-06-03 00:00:00,  window = 2004-06-09 00:00:00 -> 2005-06-02 00:00:00
pos = 2005-06-06 00:00:00,  window = 2004-06-10 00:00:00 -> 2005-06-03 00:00:00
pos = 2005-06-07 00:00:00,  window = 2004-06-11 00:00:00 -> 2005-06-06 00:00:00
pos = 2005-06-08 00:00:00,  window = 2004-06-14 00:00:00 -> 2005-06-07 00:00:00
pos = 2005-06-09 00:00:00,  window = 2004-06-15 00:00:00 -> 2005-06-08 00:00:00
pos = 2005-06-10 00:00:00,  window = 2004-06-16 00:00:00 -> 2005-06-09 00:00:00
pos = 2005-06-13 00:00:00,  window = 2004-06-17 00:00:00 -> 2005-06-10 00:00:00
pos = 2005-06-14 00:00:00,  window = 2004-06-18 00:00:00 -> 2005-06-13 00:00:00
pos = 2005-06-15 00:00:00,  window = 2004-06-21 00:00:00 -> 2005-06-14 00:00:00
pos = 2005-06-16 00:00:00,  window = 2004-06-22 00:00:00 -> 2005-06-15 00:00:00
pos = 2005-06-17 00:00:00,  window = 200

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
import matplotlib.pyplot as plt
from scipy.stats import norm

def kupiec_test(viol, alpha: float):
    # """
    # Kupiec (POF / Unconditional Coverage) Test
    # H0: P(violation)=alpha  (違規率正確)
    # viol: 0/1 序列（list/np.array/pd.Series）
    # alpha: 左尾機率，例如 0.05 或 0.01
    # 回傳 dict: n, x, viol_rate, LR_uc, p_value
    # """
    v = pd.Series(viol).dropna().astype(int).values
    n = len(v)
    if n == 0:
        raise ValueError("viol 序列為空")

    x = int(v.sum())
    p_hat = x / n

    # 避免 log(0)
    eps = 1e-12
    p_hat_c = np.clip(p_hat, eps, 1 - eps)
    a_c = np.clip(alpha, eps, 1 - eps)

    # log-likelihood under H0 and under MLE
    ll_h0  = (n - x) * np.log(1 - a_c) + x * np.log(a_c)
    ll_mle = (n - x) * np.log(1 - p_hat_c) + x * np.log(p_hat_c)

    LR_uc = -2.0 * (ll_h0 - ll_mle)
    p_val = 1.0 - chi2.cdf(LR_uc, df=1)

    return {
        "test": "Kupiec UC (POF)",
        "alpha": float(alpha),
        "n": int(n),
        "x": int(x),
        "viol_rate": float(p_hat),
        "LR_uc": float(LR_uc),
        "p_value": float(p_val),
    }


def christoffersen_cc_test(viol, alpha: float):
    # """
    # Christoffersen Conditional Coverage (CC) Test
    # H0: (1) 違規率=alpha (UC) AND (2) 違規獨立 (IND)
    # viol: 0/1 序列（list/np.array/pd.Series）
    # alpha: 左尾機率，例如 0.05 或 0.01
    # 回傳 dict: UC/IND/CC 的 LR 統計量與 p-value，以及轉移計數 n00,n01,n10,n11
    # """
    v = pd.Series(viol).dropna().astype(int).values
    n = len(v)
    if n < 2:
        raise ValueError("CC test 需要至少 2 筆資料（才能算轉移）")

    # --- 1) UC（Kupiec） ---
    uc = kupiec_test(v, alpha)
    LR_uc = uc["LR_uc"]

    # --- 2) IND（獨立性：Markov 轉移） ---
    v_lag = v[:-1]
    v_now = v[1:]

    n00 = int(np.sum((v_lag == 0) & (v_now == 0)))
    n01 = int(np.sum((v_lag == 0) & (v_now == 1)))
    n10 = int(np.sum((v_lag == 1) & (v_now == 0)))
    n11 = int(np.sum((v_lag == 1) & (v_now == 1)))

    # 轉移機率（避免除 0）
    eps = 1e-12
    pi01 = n01 / max(n00 + n01, 1)   # P(hit_t=1 | hit_{t-1}=0)
    pi11 = n11 / max(n10 + n11, 1)   # P(hit_t=1 | hit_{t-1}=1)
    pi   = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)  # 無條件（對轉移樣本 n-1）

    pi01_c = np.clip(pi01, eps, 1 - eps)
    pi11_c = np.clip(pi11, eps, 1 - eps)
    pi_c   = np.clip(pi,   eps, 1 - eps)

    # 對數概似：
    # H0(獨立)：同一個 pi
    ll_ind_h0 = (n00 + n10) * np.log(1 - pi_c) + (n01 + n11) * np.log(pi_c)

    # H1(一階 Markov)：兩個轉移機率 pi01, pi11
    ll_ind_h1 = (n00) * np.log(1 - pi01_c) + (n01) * np.log(pi01_c) \
              + (n10) * np.log(1 - pi11_c) + (n11) * np.log(pi11_c)

    LR_ind = -2.0 * (ll_ind_h0 - ll_ind_h1)
    p_ind  = 1.0 - chi2.cdf(LR_ind, df=1)

    # --- 3) CC（條件覆蓋） ---
    LR_cc = LR_uc + LR_ind
    p_cc  = 1.0 - chi2.cdf(LR_cc, df=2)

    return {
        "test": "Christoffersen CC",
        "alpha": float(alpha),
        "n": int(n),
        "x": int(np.sum(v)),
        "viol_rate": float(np.mean(v)),
        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "LR_uc": float(LR_uc),
        "p_uc": float(uc["p_value"]),
        "LR_ind": float(LR_ind),
        "p_ind": float(p_ind),
        "LR_cc": float(LR_cc),
        "p_cc": float(p_cc),
    }


# =======================
# 用 df_export 直接跑
# =======================
# VaR 95%：alpha=0.05
kupiec_95 = kupiec_test(df_export["viol_95"], alpha=0.05)
cc_95     = christoffersen_cc_test(df_export["viol_95"], alpha=0.05)

# VaR 99%：alpha=0.01
kupiec_99 = kupiec_test(df_export["viol_99"], alpha=0.01)
cc_99     = christoffersen_cc_test(df_export["viol_99"], alpha=0.01)


def build_backtest_summary(kupiec_dict, cc_dict, level_tag: str):
    # """
    # level_tag: '95' or '99'
    # """
    alpha = float(kupiec_dict["alpha"])
    n = int(kupiec_dict["n"])
    x = int(kupiec_dict["x"])
    viol_rate = float(kupiec_dict["viol_rate"])
    expected_viol = alpha * n

    return {
        "level": level_tag,
        "alpha": alpha,
        "n": n,
        "x": x,
        "viol_rate": viol_rate,
        "expected_viol": expected_viol,

        "LR_uc": float(kupiec_dict["LR_uc"]),
        "p_uc": float(kupiec_dict["p_value"]),

        "LR_ind": float(cc_dict["LR_ind"]),
        "p_ind": float(cc_dict["p_ind"]),

        "LR_cc": float(cc_dict["LR_cc"]),
        "p_cc": float(cc_dict["p_cc"]),

        "pass_uc_5pct": float(kupiec_dict["p_value"]) > 0.05,
        "pass_cc_5pct": float(cc_dict["p_cc"]) > 0.05,
    }


def build_transition_row(cc_dict, level_tag: str):
    # """
    # 把 n00 n01 n10 n11 與轉移機率 pi01 pi11 存起來，方便看群聚。
    # """
    n00 = int(cc_dict["n00"])
    n01 = int(cc_dict["n01"])
    n10 = int(cc_dict["n10"])
    n11 = int(cc_dict["n11"])

    # 轉移機率（避免除 0）
    denom0 = (n00 + n01)
    denom1 = (n10 + n11)
    pi01 = n01 / denom0 if denom0 > 0 else float("nan")
    pi11 = n11 / denom1 if denom1 > 0 else float("nan")

    return {
        "level": level_tag,
        "alpha": float(cc_dict["alpha"]),
        "n": int(cc_dict["n"]),
        "x": int(cc_dict["x"]),
        "viol_rate": float(cc_dict["viol_rate"]),

        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "pi01": pi01,
        "pi11": pi11,
        "pi11_minus_pi01": (pi11 - pi01) if (pd.notna(pi11) and pd.notna(pi01)) else float("nan"),

        "LR_ind": float(cc_dict["LR_ind"]),
        "p_ind": float(cc_dict["p_ind"]),
    }


# ====== 組表 ======
summary_rows = [
    build_backtest_summary(kupiec_95, cc_95, "95"),
    build_backtest_summary(kupiec_99, cc_99, "99"),
]
df_summary = pd.DataFrame(summary_rows)

transition_rows = [
    build_transition_row(cc_95, "95"),
    build_transition_row(cc_99, "99"),
]
df_transition = pd.DataFrame(transition_rows)

# ====== 存檔 ======
# output_dir = "./LSTM_diagnostics"
# os.makedirs(output_dir, exist_ok=True)

backtest_path = os.path.join(output_dir, "backtest_summary.csv")
trans_path    = os.path.join(output_dir, "transition_matrix.csv")

df_summary.to_csv(backtest_path, index=False, encoding="utf-8-sig")
df_transition.to_csv(trans_path, index=False, encoding="utf-8-sig")

print(f"✅ saved: {backtest_path}")
print(f"✅ saved: {trans_path}")

print("\n--- backtest_summary ---")
print(df_summary)

print("\n--- transition_matrix ---")
print(df_transition)



# =========================
# 1) UC 覆蓋率圖（含 95% CI）
# =========================
def wilson_ci(x: int, n: int, z: float = 1.96):
    # """
    # Wilson score interval for binomial proportion.
    # 回傳 (low, high)
    # """
    if n <= 0:
        return (np.nan, np.nan)
    p = x / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z * np.sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)

def plot_uc_coverage(df_export: pd.DataFrame, out_path: str = "uc_coverage.png"):
    # 兩個水準：95% VaR -> alpha=0.05；99% VaR -> alpha=0.01
    specs = [
        ("95%", 0.05, "viol_95"),
        ("99%", 0.01, "viol_99"),
    ]

    rows = []
    for level, alpha, col in specs:
        v = df_export[col].dropna().astype(int).values
        n = len(v)
        x = int(v.sum())
        p_hat = x / n if n > 0 else np.nan
        ci_low, ci_high = wilson_ci(x, n, z=1.96)
        rows.append((level, alpha, n, x, p_hat, ci_low, ci_high))

    res = pd.DataFrame(rows, columns=["level", "alpha", "n", "x", "viol_rate", "ci_low", "ci_high"])

    # 畫圖
    fig, ax = plt.subplots(figsize=(7, 4))
    xs = np.arange(len(res))

    # 點 + 誤差棒（違規率估計與 95% CI）
    y = res["viol_rate"].values
    yerr = np.vstack([y - res["ci_low"].values, res["ci_high"].values - y])
    ax.errorbar(xs, y, yerr=yerr, fmt='o', capsize=4, label="Observed violation rate (95% CI)")

    # 理論線：alpha
    ax.plot(xs, res["alpha"].values, linestyle='--', marker='s', label="Theoretical alpha")

    ax.set_xticks(xs)
    ax.set_xticklabels([f"{lv}\n(n={n}, x={x})" for lv, n, x in zip(res["level"], res["n"], res["x"])])
    ax.set_ylabel("Violation Rate")
    ax.set_title("Unconditional Coverage (UC): Observed vs Theoretical")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    return res  # 若你想順便存成表也方便


# =========================
# 2) Hit timeline（違規時間線）
# =========================
def plot_hit_timeline(df_export: pd.DataFrame, out_path: str = "hit_timeline.png"):
    # """
    # 畫 95% 與 99% 的違規時間點（1 的日期），上下兩條軸並排。
    # """
    if not isinstance(df_export.index, pd.DatetimeIndex):
        # 若 DATE 是欄位而不是 index
        if "DATE" in df_export.columns:
            df_export = df_export.set_index(pd.to_datetime(df_export["DATE"]))
        else:
            raise ValueError("df_export 需要 DatetimeIndex，或包含 DATE 欄位。")

    v95 = df_export["viol_95"].fillna(0).astype(int)
    v99 = df_export["viol_99"].fillna(0).astype(int)

    dates = df_export.index

    fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)

    # 95% hit
    hit_dates_95 = dates[v95.values == 1]
    axes[0].eventplot(hit_dates_95, lineoffsets=1, linelengths=0.8)
    axes[0].set_yticks([1])
    axes[0].set_yticklabels(["VaR 95% hit"])
    axes[0].set_title("Hit Timeline (Violations Over Time)")
    axes[0].grid(True, axis='x', alpha=0.3)

    # 99% hit
    hit_dates_99 = dates[v99.values == 1]
    axes[1].eventplot(hit_dates_99, lineoffsets=1, linelengths=0.8)
    axes[1].set_yticks([1])
    axes[1].set_yticklabels(["VaR 99% hit"])
    axes[1].grid(True, axis='x', alpha=0.3)

    axes[1].set_xlabel("Time")

    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)


# =========================
# 直接呼叫（你只要確保 df_export 已存在）
# =========================
uc_table = plot_uc_coverage(df_export, out_path="./LSTM_diagnostics/uc_coverage.png")
plot_hit_timeline(df_export, out_path="./LSTM_diagnostics/hit_timeline.png")

# （可選）把 UC 結果表也存起來，論文附錄很好用
uc_table.to_csv("./LSTM_diagnostics/uc_coverage_table.csv", index=False, encoding="utf-8-sig")
print("Saved: uc_coverage.png, hit_timeline.png, uc_coverage_table.csv")


# df_export 的 index 應該是 DATE
dfp = df_export.copy()
dfp.index = pd.to_datetime(dfp.index)

fig_path = f"./LSTM_diagnostics/{stem}_VaR_price{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"  # 你原本存圖路徑變數

plt.figure(figsize=(12, 6))

plt.plot(dfp.index, dfp['price_true'], linewidth=0.9, alpha=0.9, label='Price (True)')

viol = dfp['viol_95'].fillna(0).astype(int).astype(bool)

ymin = float(dfp['price_true'].min())
ymax = float(dfp['price_true'].max())

plt.vlines(dfp.index[viol], ymin=ymin, ymax=ymax, color='red', alpha=0.25, linewidth=0.6,
           label='VaR 95% Violation Day')

plt.title(f"{ASSET_SYMBOL_ES1} Price with VaR 95% Violations (Vertical Lines)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
# # 1) 價格主線
# plt.plot(dfp.index, dfp['price_true'], linewidth=0.9, alpha=0.9, label='Price (True)')

# # 2) 95% 違規日（viol_95 已經在 df_export）
# viol = dfp['viol_95'].fillna(0).astype(int).astype(bool)

# # 3) 違規日上色（透明度高一點 -> alpha 大一點）
# shade_alpha = 0.35
# ax = plt.gca()

# for d in dfp.index[viol]:
#     ax.axvspan(d, d + pd.Timedelta(days=1), color='red', alpha=shade_alpha, linewidth=0)

# plt.title(f"{ASSET_SYMBOL_ES1} Price with VaR 95% Violations (Shaded)")
# plt.xlabel("Time")
# plt.ylabel("Price")
# plt.grid(True, alpha=0.3)
# plt.legend()

# plt.savefig(fig_path, dpi=300, bbox_inches='tight')
# plt.close()



✅ saved: ./LSTM_diagnostics/backtest_summary.csv
✅ saved: ./LSTM_diagnostics/transition_matrix.csv

--- backtest_summary ---
  level  alpha     n    x  viol_rate  expected_viol      LR_uc      p_uc  \
0    95   0.05  5072  309   0.060923         253.60  11.945750  0.000548   
1    99   0.01  5072   83   0.016364          50.72  17.406332  0.000030   

     LR_ind     p_ind      LR_cc      p_cc  pass_uc_5pct  pass_cc_5pct  
0  0.986878  0.320507  12.932628  0.001555         False         False  
1  0.274247  0.600497  17.680579  0.000145         False         False  

--- transition_matrix ---
  level  alpha     n    x  viol_rate   n00  n01  n10  n11      pi01      pi11  \
0    95   0.05  5072  309   0.060923  4476  286  286   23  0.060059  0.074434   
1    99   0.01  5072   83   0.016364  4907   81   81    2  0.016239  0.024096   

   pi11_minus_pi01    LR_ind     p_ind  
0         0.014375  0.986878  0.320507  
1         0.007857  0.274247  0.600497  
Saved: uc_coverage.png, hit_timel